In [11]:
import os

In [12]:
%pwd

'C:\\Users\\Dell User\\Desktop\\DeepLearm\\Deep_Learning'

In [13]:
os.chdir(r"C:\Users\Dell User\Desktop\DeepLearm\Deep_Learning")

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [15]:
# configuration manager
from src.constants import *
from src.utils.common import read_yaml,create_directories
import tensorflow as tf 

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir,
            "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [7]:
import os 
import tensorflow as tf 
from zipfile import ZipFile
import urllib.request as request



class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
    def get_base_model(self):
        self.model=tf.keras.models.load_model(self.config.updated_base_model_path)
  
    def train_valid_generator(self):
        from tensorflow.keras.applications.densenet import preprocess_input

        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input, # <-- CRITICAL: Align input distribution
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=10,
                horizontal_flip=True,
                vertical_flip=True, 
                width_shift_range=0.2,
                height_shift_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

        callbacks = [
            EarlyStopping(
                monitor="val_accuracy",
                patience=15,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_accuracy",
                factor=0.5,
                patience=7,
                min_lr=1e-7
            ),
            ModelCheckpoint(
                filepath=str(self.config.trained_model_path),  # saves to your config path
                monitor="val_accuracy",
                save_best_only=True
            )
        ]
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callbacks
          
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [16]:
import os
import math
import numpy as np
import tensorflow as tf
from pathlib import Path


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self, model_path: Path = None):
        path = model_path or self.config.updated_base_model_path
        self.model = tf.keras.models.load_model(path)

    def train_valid_generator(self):
        from tensorflow.keras.applications.densenet import preprocess_input

        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical"  # explicit, since you have 4 classes
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=10,
                horizontal_flip=True,
                vertical_flip=False,   # anatomically implausible for CT, removed
                width_shift_range=0.1,
                height_shift_range=0.1,
                zoom_range=0.1,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

        # Compute class weights from the actual training generator's label distribution
        self.class_weights = self._compute_class_weights()

    def _compute_class_weights(self):
        """
        Inverse-frequency class weights based on your actual counts:
        Normal: 5077, Cyst: 3709, Tumor: 2283, Stone: 1377
        Computed dynamically here so it stays correct even if your data changes.
        """
        from sklearn.utils.class_weight import compute_class_weight

        classes = self.train_generator.classes  # array of integer labels for every training sample
        class_indices = self.train_generator.class_indices  # e.g. {'Cyst':0,'Normal':1,'Stone':2,'Tumor':3}

        unique_classes = np.unique(classes)
        weights = compute_class_weight(
            class_weight="balanced",
            classes=unique_classes,
            y=classes
        )

        class_weight_dict = dict(zip(unique_classes, weights))
        print("Class indices:", class_indices)
        print("Computed class weights:", class_weight_dict)
        return class_weight_dict

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self, epochs: int = None, save_path: Path = None, use_class_weights: bool = True):
        from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

        target_path = save_path or self.config.trained_model_path

        callbacks = [
            EarlyStopping(
                monitor="val_accuracy",
                patience=15,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_accuracy",
                factor=0.5,
                patience=7,
                min_lr=1e-7
            ),
            ModelCheckpoint(
                filepath=str(target_path),
                monitor="val_accuracy",
                save_best_only=True
            )
        ]

        self.steps_per_epoch = math.ceil(self.train_generator.samples / self.train_generator.batch_size)
        self.validation_steps = math.ceil(self.valid_generator.samples / self.valid_generator.batch_size)

        fit_kwargs = dict(
            x=self.train_generator,
            epochs=epochs or self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            callbacks=callbacks
        )

        if use_class_weights:
            fit_kwargs["class_weight"] = self.class_weights

        self.model.fit(**fit_kwargs)

        self.save_model(
            path=target_path,
            model=self.model
        )

In [10]:
import os

# Use the actual folder path on your disk
data_dir = "artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"

for cls in os.listdir(data_dir):
    cls_path = os.path.join(data_dir, cls)
    if os.path.isdir(cls_path):
        print(cls, len(os.listdir(cls_path)))


Cyst 3709
Normal 5077
Stone 1377
Tumor 2283


In [ ]:

try:
    config=ConfigurationManager()
    training_config =config.get_training_config()
    training = Training(training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

In [17]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-06-28 19:19:27,627: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 19:19:27,631: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 19:19:27,634: INFO: common: created directory at: artifacts]
[2026-06-28 19:19:27,635: INFO: common: created directory at: artifacts\training]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
Class indices: {'Cyst': 0, 'Normal': 1, 'Stone': 2, 'Tumor': 3}
Computed class weights: {0: 0.8388645552560647, 1: 0.6129369768586903, 2: 2.259301270417423, 3: 1.3627531472359058}
Epoch 1/100
623/623 [==============================] - 152s 235ms/step - loss: 0.9680 - accuracy: 0.6401 - val_loss: 1.1006 - val_accuracy: 0.5770 - lr: 1.0000e-04
Epoch 2/100
623/623 [==============================] - 142s 228ms/step - loss: 0.5650 - accuracy: 0.7921 - val_loss: 1.2426 - val_accuracy: 0.5336 - lr: 1.0000e-04
Epoch 3/100
623/623 [==============================] - 151s 242ms/step - loss:

KeyboardInterrupt: 

In [10]:
import tensorflow as tf
print("Is GPU available? ", tf.config.list_physical_devices('GPU'))


Is GPU available?  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
import sys
import tensorflow as tf

print("Python path:", sys.executable)
print("TensorFlow version:", tf.__version__)
print("Is GPU available? ", tf.config.list_physical_devices('GPU'))


Python path: c:\Users\Dell User\Desktop\DeepLearm\Deep_Learning\.venv\Scripts\python.exe
TensorFlow version: 2.10.0
Is GPU available?  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
